# **Loading the Raw Data**

---

First pass over the three CMS source files this project is built on: physician
group MIPS performance, the clinician and organization directory (DAC), and
county level Medicare outcomes (Geographic Variation). Each file is loaded and
profiled for shape, dtypes, missingness, and uniqueness so the crosswalk in
notebook 02 starts from a known baseline.

**Setup:** Pinned dependencies: `python -m pip install -r ../requirements.txt`

## Table of Contents

1. [Setup and load raw files](#s1)
2. [MIPS exploration](#s2)
3. [DAC exploration](#s3)
4. [GEO exploration](#s4)

---
<a id="s1"></a>

## **1. Setup and load raw files**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Display all columns and use a wider table width
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Load the three raw CMS files
PROJECT_DIR = Path.cwd().parent
RAW_DATA = PROJECT_DIR / "data" / "raw"

mips = pd.read_csv(RAW_DATA / "grp_public_reporting.csv")
dac = pd.read_csv(RAW_DATA / "DAC_NationalDownloadableFile.csv")
geo = pd.read_csv(RAW_DATA / "2014-2024 Original Medicare Geographic Variation Public Use File.csv")

# MIPS column names come with leading spaces
mips.columns = mips.columns.str.strip()

/var/folders/bc/lrkcgdlx3332x9brp_np1l_c0000gn/T/ipykernel_77606/1452285722.py:15: DtypeWarning: Columns (9,14) have mixed types. Specify dtype option on import or set low_memory=False.
  mips = pd.read_csv(RAW_DATA / "grp_public_reporting.csv")


/var/folders/bc/lrkcgdlx3332x9brp_np1l_c0000gn/T/ipykernel_77606/1452285722.py:16: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  dac = pd.read_csv(RAW_DATA / "DAC_NationalDownloadableFile.csv")


/var/folders/bc/lrkcgdlx3332x9brp_np1l_c0000gn/T/ipykernel_77606/1452285722.py:17: DtypeWarning: Columns (222,223) have mixed types. Specify dtype option on import or set low_memory=False.
  geo = pd.read_csv(RAW_DATA / "2014-2024 Original Medicare Geographic Variation Public Use File.csv")


---
<a id="s2"></a>

## **2. MIPS exploration**

`grp_public_reporting.csv`: physician group MIPS quality measures and performance rates.

In [2]:
# Quick look
print('shape:', mips.shape)
display(mips.head())
display(mips.dtypes.to_frame('dtype'))

shape: (199228, 16)


,Facility Name,org_PAC_ID,ACO_ID_1,ACO_nm_1,ACO_ID_2,ACO_nm_2,measure_cd,measure_title,invs_msr,attestation_value,prf_rate,patient_count,star_value,five_star_benchmark,collection_type,CCXP_ind
0,CLINT VANLANDINGHAM,42246605,NaN,NaN,NaN,NaN,IA_GRP_AHE_1,Enhance Engagement of Medicaid and Other Under...,N,Y,NaN,NaN,NaN,NaN,NaN,Y
1,MOBILE MBS INC,42357527,NaN,NaN,NaN,NaN,IA_GRP_AHE_1,Enhance Engagement of Medicaid and Other Under...,N,Y,NaN,NaN,NaN,NaN,NaN,Y
2,ORLANDO GALINDEZ MD PA,42483646,NaN,NaN,NaN,NaN,IA_GRP_AHE_1,Enhance Engagement of Medicaid and Other Under...,N,Y,NaN,NaN,NaN,NaN,NaN,Y
3,KARPIK AND RICE EYECARE PC,42534281,NaN,NaN,NaN,NaN,IA_GRP_AHE_1,Enhance Engagement of Medicaid and Other Under...,N,Y,NaN,NaN,NaN,NaN,NaN,Y
4,"TENNESSEE EYE CARE SPECIALISTS, PLLC",42544751,NaN,NaN,NaN,NaN,IA_GRP_AHE_1,Enhance Engagement of Medicaid and Other Under...,N,Y,NaN,NaN,NaN,NaN,NaN,Y


,dtype
Facility Name,object
org_PAC_ID,int64
ACO_ID_1,object
ACO_nm_1,object
ACO_ID_2,float64
ACO_nm_2,float64
measure_cd,object
measure_title,object
invs_msr,object
attestation_value,object


In [3]:
# Profile: structure, distributions, missingness, uniqueness
mips.info()

print('\ndescribe:')
display(mips.describe())

print('\nmissing values per column:')
display(mips.isna().sum())

print('\nunique values per column:')
display(mips.nunique())

print('\nmeasure_cd value counts:')
display(mips.measure_cd.value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199228 entries, 0 to 199227
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Facility Name        199228 non-null  object 
 1   org_PAC_ID           199228 non-null  int64  
 2   ACO_ID_1             24878 non-null   object 
 3   ACO_nm_1             24878 non-null   object 
 4   ACO_ID_2             0 non-null       float64
 5   ACO_nm_2             0 non-null       float64
 6   measure_cd           199228 non-null  object 
 7   measure_title        199228 non-null  object 
 8   invs_msr             199228 non-null  object 
 9   attestation_value    87117 non-null   object 
 10  prf_rate             112111 non-null  float64
 11  patient_count        107322 non-null  float64
 12  star_value           70455 non-null   float64
 13  five_star_benchmark  70455 non-null   float64
 14  collection_type      112111 non-null  object 
 15  CCXP_ind         

,org_PAC_ID,ACO_ID_2,ACO_nm_2,prf_rate,patient_count,star_value,five_star_benchmark
count,1.992280e+05,0.0,0.0,112111.000000,1.073220e+05,70455.000000,70455.000000
mean,4.920434e+09,NaN,NaN,57.421312,9.147178e+03,3.392932,92.894301
std,2.868212e+09,NaN,NaN,39.497243,4.840461e+04,1.352603,10.458832
min,4.210037e+07,NaN,NaN,0.000000,2.000000e+01,1.000000,34.000000
25%,2.466443e+09,NaN,NaN,14.000000,2.880000e+02,2.000000,86.000000
50%,4.880788e+09,NaN,NaN,69.000000,1.221000e+03,4.000000,99.000000
75%,7.315936e+09,NaN,NaN,97.000000,4.652750e+03,4.000000,100.000000
max,9.931590e+09,NaN,NaN,100.000000,4.450932e+06,5.000000,100.000000



missing values per column:


Facility Name               0
org_PAC_ID                  0
ACO_ID_1               174350
ACO_nm_1               174350
ACO_ID_2               199228
ACO_nm_2               199228
measure_cd                  0
measure_title               0
invs_msr                    0
attestation_value      112111
prf_rate                87117
patient_count           91906
star_value             128773
five_star_benchmark    128773
collection_type         87117
CCXP_ind                    0
dtype: int64


unique values per column:


Facility Name          14859
org_PAC_ID             14976
ACO_ID_1                 402
ACO_nm_1                 400
ACO_ID_2                   0
ACO_nm_2                   0
measure_cd               367
measure_title            341
invs_msr                   2
attestation_value          2
prf_rate                 240
patient_count          20301
star_value                 5
five_star_benchmark       29
collection_type            6
CCXP_ind                   2
dtype: int64


measure_cd value counts:


measure_cd
PI_GRP_ONCDIR_1         5293
PI_GRP_PPHI_1           5263
PI_GRP_PEA_1            5209
IA_GRP_EPA_1            5204
MIPS_GRP_130_overall    4507
                        ... 
IA_GRP_BMH_10             17
IA_GRP_PSPA_33            15
PI_GRP_PHCDRR_1_EX_3      15
IA_GRP_PSPA_3             13
PI_GRP_INFBLO_1            1
Name: count, Length: 367, dtype: int64

---
<a id="s3"></a>

## **3. DAC exploration**

`DAC_NationalDownloadableFile.csv`: clinician and organization directory used to map groups to ZIP codes.

In [4]:
# Quick look
print('shape:', dac.shape)
display(dac.head())
print('columns:', list(dac.columns))

shape: (3387942, 31)


,NPI,Ind_PAC_ID,Ind_enrl_ID,Provider Last Name,Provider First Name,Provider Middle Name,suff,gndr,Cred,Med_sch,Grd_yr,pri_spec,sec_spec_1,sec_spec_2,sec_spec_3,sec_spec_4,sec_spec_all,Telehlth,Facility Name,org_pac_id,num_org_mem,adr_ln_1,adr_ln_2,ln_2_sprs,City/Town,State,ZIP Code,Telephone Number,ind_assgn,grp_assgn,adrs_id
0,1235888272,2365958196,I20250718004023,BAEZ MUNIZ,EDUARDO,NaN,NaN,M,NaN,OTHER,1984.0,CLINICAL PSYCHOLOGIST,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AGUADA,PR,602,9.392995e+09,Y,M,PR00602XXXXAGXXXXXXXXXX00
1,1568251114,1153817721,I20260102002504,FULGENCIO,TOSCANIA,Y.,NaN,F,MD,OTHER,2012.0,GENERAL PRACTICE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AGUADA,PR,602,7.875890e+09,Y,M,PR00602XXXXAGXXXXXXXXXX00
2,1427629856,446726608,I20260415004193,MORALES RODRIGUEZ,ANA,L,NaN,F,MD,OTHER,2012.0,GENERAL PRACTICE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AGUADILLA,PR,603,4.075438e+09,Y,M,PR00603XXXXAGXXXXXXXXXX00
3,1578448098,8921583808,I20260329000120,TROCHE,MARISABEL,NaN,NaN,U,CSW,OTHER,2024.0,CLINICAL SOCIAL WORKER,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BOQUERON,PR,622,7.873181e+09,Y,M,PR00622XXXXBOXXXXXXXXXX00
4,1205780111,42786774,I20260420001601,MIRANDA,ISAMAR,NaN,NaN,F,NaN,OTHER,2025.0,REGISTERED DIETITIAN OR NUTRITION PROFESSIONAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DORADO,PR,646,7.872436e+09,Y,M,PR00646XXXXDOXXXXXXXXXX00


columns: ['NPI', 'Ind_PAC_ID', 'Ind_enrl_ID', 'Provider Last Name', 'Provider First Name', 'Provider Middle Name', 'suff', 'gndr', 'Cred', 'Med_sch', 'Grd_yr', 'pri_spec', 'sec_spec_1', 'sec_spec_2', 'sec_spec_3', 'sec_spec_4', 'sec_spec_all', 'Telehlth', 'Facility Name', 'org_pac_id', 'num_org_mem', 'adr_ln_1', 'adr_ln_2', 'ln_2_sprs', 'City/Town', 'State', 'ZIP Code', 'Telephone Number', 'ind_assgn', 'grp_assgn', 'adrs_id']


In [5]:
# Profile: structure, distributions, missingness, duplicates, uniqueness
dac.info()

print('\ndescribe:')
display(dac.describe())

print('\nmissing values per column:')
display(dac.isna().sum())

print('\nduplicated rows:', dac.duplicated().sum())

print('\nunique values per column:')
display(dac.nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3387942 entries, 0 to 3387941
Data columns (total 31 columns):
 #   Column                Dtype  
---  ------                -----  
 0   NPI                   int64  
 1   Ind_PAC_ID            int64  
 2   Ind_enrl_ID           object 
 3   Provider Last Name    object 
 4   Provider First Name   object 
 5   Provider Middle Name  object 
 6   suff                  object 
 7   gndr                  object 
 8   Cred                  object 
 9   Med_sch               object 
 10  Grd_yr                float64
 11  pri_spec              object 
 12  sec_spec_1            object 
 13  sec_spec_2            object 
 14  sec_spec_3            object 
 15  sec_spec_4            object 
 16  sec_spec_all          object 
 17  Telehlth              object 
 18  Facility Name         object 
 19  org_pac_id            float64
 20  num_org_mem           float64
 21  adr_ln_1              object 
 22  adr_ln_2              object 
 23  ln_2_sp

,NPI,Ind_PAC_ID,Grd_yr,org_pac_id,num_org_mem,Telephone Number
count,3.387942e+06,3.387942e+06,3.385285e+06,3.043125e+06,3.043124e+06,2.857364e+06
mean,1.499760e+09,4.997535e+09,2.007685e+03,4.943616e+09,1.042062e+03,1.722865e+12
std,2.876871e+08,2.872203e+09,1.242748e+01,2.852380e+09,1.729164e+03,8.801043e+13
min,1.003000e+09,4.210004e+07,1.949000e+03,4.210009e+07,2.000000e+00,2.003745e+09
25%,1.255001e+09,2.466986e+09,1.999000e+03,2.567372e+09,5.800000e+01,4.016494e+09
50%,1.497954e+09,4.981982e+09,2.010000e+03,4.789597e+09,3.130000e+02,6.123339e+09
75%,1.740923e+09,7.416975e+09,2.018000e+03,7.416842e+09,1.371000e+03,8.084860e+09
max,1.993000e+09,9.931700e+09,2.035000e+03,9.931697e+09,1.124400e+04,9.496501e+15



missing values per column:


NPI                           0
Ind_PAC_ID                    0
Ind_enrl_ID                   5
Provider Last Name           82
Provider First Name          49
Provider Middle Name    1231919
suff                    3347361
gndr                          0
Cred                     559953
Med_sch                      54
Grd_yr                     2657
pri_spec                      5
sec_spec_1              3007095
sec_spec_2              3348843
sec_spec_3              3383737
sec_spec_4              3387322
sec_spec_all            3007095
Telehlth                2585517
Facility Name            344817
org_pac_id               344817
num_org_mem              344818
adr_ln_1                  21726
adr_ln_2                2250198
ln_2_sprs               3171598
City/Town                     0
State                         0
ZIP Code                      0
Telephone Number         530578
ind_assgn                     0
grp_assgn                     0
adrs_id                       0
dtype: i


duplicated rows: 0

unique values per column:


NPI                     1616566
Ind_PAC_ID              1616600
Ind_enrl_ID             1742970
Provider Last Name       338421
Provider First Name      112587
Provider Middle Name      55410
suff                         11
gndr                          3
Cred                         22
Med_sch                     468
Grd_yr                       84
pri_spec                    100
sec_spec_1                   81
sec_spec_2                   68
sec_spec_3                   58
sec_spec_4                   40
sec_spec_all               1593
Telehlth                      1
Facility Name             82664
org_pac_id                83321
num_org_mem                 850
adr_ln_1                 303562
adr_ln_2                  36093
ln_2_sprs                     1
City/Town                 16684
State                        56
ZIP Code                 334224
Telephone Number         329831
ind_assgn                     2
grp_assgn                     2
adrs_id                  458933
dtype: i

In [6]:
# Map each physician group (org_pac_id) to the ZIP codes it appears under
zip_by_pac = dac.groupby('org_pac_id')['ZIP Code'].unique().reset_index()
zip_by_pac['zip_count'] = zip_by_pac['ZIP Code'].apply(len)

print('Total physician groups:', len(zip_by_pac))
print('groups with 1 ZIP code: ', (zip_by_pac['zip_count'] == 1).sum())
print('groups with >1 ZIP codes:', (zip_by_pac['zip_count'] > 1).sum())

pct = (zip_by_pac['zip_count'] > 1).mean() * 100
print(f'Percentage with multiple ZIPs: {pct:.1f}%')

display(zip_by_pac.head(20))

Total physician groups: 83321
groups with 1 ZIP code:  53371
groups with >1 ZIP codes: 29950
Percentage with multiple ZIPs: 35.9%


,org_pac_id,ZIP Code,zip_count
0,42100091.0,"[372102745, 372075408, 370876709, 372112094, 3...",12
1,42100372.0,"[974774379, 974772594, 974087300]",3
2,42100513.0,[115981739],1
3,42100521.0,[972191940],1
4,42101081.0,[809201094],1
5,42101099.0,"[42406069, 042407415]",2
6,42101347.0,"[189541108, 191162151, 190471652, 189401877, 1...",6
7,42101560.0,[801112528],1
8,42101982.0,"[628393241, 628581053, 628241113]",3
9,42102170.0,"[280252440, 280252457, 280252927]",3


---
<a id="s4"></a>

## **4. GEO exploration**

`2014-2024 Original Medicare Geographic Variation Public Use File.csv`: county level Medicare utilization and outcomes.

In [7]:
# Quick look
print('shape:', geo.shape)
print('columns:', list(geo.columns))

shape: (36994, 246)
columns: ['YEAR', 'BENE_GEO_LVL', 'BENE_GEO_DESC', 'BENE_GEO_CD', 'BENE_AGE_LVL', 'BENES_TOTAL_CNT', 'BENES_WTH_PTAPTB_CNT', 'BENES_OM_CNT', 'BENES_MA_CNT', 'MA_PRTCPTN_RATE', 'BENE_AVG_AGE', 'BENE_FEML_PCT', 'BENE_MALE_PCT', 'BENE_RACE_WHT_PCT', 'BENE_RACE_BLACK_PCT', 'BENE_RACE_HSPNC_PCT', 'BENE_RACE_OTHR_PCT', 'BENE_DUAL_PCT', 'TOT_MDCR_PYMT_AMT', 'TOT_MDCR_STDZD_PYMT_AMT', 'TOT_MDCR_PYMT_PC', 'TOT_MDCR_STDZD_PYMT_PC', 'IP_MDCR_PYMT_AMT', 'IP_MDCR_PYMT_PCT', 'IP_MDCR_PYMT_PC', 'IP_MDCR_PYMT_PER_USER', 'IP_MDCR_STDZD_PYMT_AMT', 'IP_MDCR_STDZD_PYMT_PCT', 'IP_MDCR_STDZD_PYMT_PC', 'IP_MDCR_STDZD_PYMT_PER_USER', 'BENES_IP_CVRD_STAY_CNT', 'BENES_IP_PCT', 'IP_CVRD_STAYS_PER_1000_BENES', 'IP_CVRD_DAYS_PER_1000_BENES', 'ACUTE_HOSP_READMSN_CNT', 'ACUTE_HOSP_READMSN_PCT', 'BENES_ER_VISITS_CNT', 'ER_VISITS_PER_1000_BENES', 'BENES_ER_VISITS_PCT', 'OP_MDCR_PYMT_AMT', 'OP_MDCR_PYMT_PCT', 'OP_MDCR_PYMT_PC', 'OP_MDCR_PYMT_PER_USER', 'OP_MDCR_STDZD_PYMT_AMT', 'OP_MDCR_STDZD_PYMT_P

In [8]:
# Profile: structure, missingness, duplicates, uniqueness, distributions
geo.info()

print('\nmissing values per column:')
display(geo.isna().sum())

print('\nduplicated rows:', geo.duplicated().sum())

print('\nunique values per column:')
display(geo.nunique())

print('\ndescribe:')
display(geo.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36994 entries, 0 to 36993
Columns: 246 entries, YEAR to PQI16_LWRXTRMTY_AMPUTN_AGE_GE_75
dtypes: float64(1), int64(1), object(244)
memory usage: 69.4+ MB

missing values per column:


YEAR                                    0
BENE_GEO_LVL                            0
BENE_GEO_DESC                           0
BENE_GEO_CD                            99
BENE_AGE_LVL                            0
                                    ...  
PQI12_UTI_AGE_GE_75                 35762
PQI15_ASTHMA_AGE_LT_40              35762
PQI16_LWRXTRMTY_AMPUTN_AGE_LT_65    35762
PQI16_LWRXTRMTY_AMPUTN_AGE_65_74    35762
PQI16_LWRXTRMTY_AMPUTN_AGE_GE_75    35762
Length: 246, dtype: int64


duplicated rows: 0

unique values per column:


YEAR                                  11
BENE_GEO_LVL                           3
BENE_GEO_DESC                       3264
BENE_GEO_CD                         3261
BENE_AGE_LVL                           3
                                    ... 
PQI12_UTI_AGE_GE_75                  437
PQI15_ASTHMA_AGE_LT_40               194
PQI16_LWRXTRMTY_AMPUTN_AGE_LT_65     232
PQI16_LWRXTRMTY_AMPUTN_AGE_65_74      91
PQI16_LWRXTRMTY_AMPUTN_AGE_GE_75      83
Length: 246, dtype: int64


describe:


,YEAR,BENE_GEO_CD
count,36994.000000,36895.000000
mean,2019.000973,28989.969833
std,3.162551,16227.632746
min,2014.000000,1.000000
25%,2016.000000,17145.000000
50%,2019.000000,29021.000000
75%,2022.000000,45003.000000
max,2024.000000,78030.000000


---
[Next: ZIP to County Crosswalk &#8594;](02_zip_county_crosswalk.ipynb)